# Dropsonde Vertical Velocity vs IMERG Precipitation

Generates comparison figures: raw BEACH ω + M4+ERA5 ω profile (left) vs IMERG 30-min snapshots during the circle window (right).

**Three run modes — pick one cell to execute:**
1. Single circle by number
2. All circles on a specific date
3. All 89 circles (save only)

In [ ]:
import sys
sys.path.insert(0, '..')

from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib
matplotlib.use('Agg')

from scripts.config import (
    default_zarr_path,
    default_categories_csv_path,
    default_imerg_combined_path,
)
from scripts.imerg_only_comparison import (
    plot_one_circle,
    build_circle_sonde_index,
    imerg_times_as_np64,
)

OUT_DIR = Path('../outputs/figures/week3')
OUT_DIR.mkdir(parents=True, exist_ok=True)

CAT_COL = 'category_plane'  # or 'category_avg' / 'category_evolutionary'

print('Loading datasets ...')
ds_sonde       = xr.open_zarr(default_zarr_path())
df_cats        = pd.read_csv(default_categories_csv_path())
ds_imerg       = xr.open_dataset(default_imerg_combined_path())
imerg_times_np = imerg_times_as_np64(ds_imerg)
cum_index      = build_circle_sonde_index(ds_sonde)
circles        = ds_sonde['circle'].values
ctimes         = pd.to_datetime(ds_sonde['circle_time'].values)
print(f'Ready — {len(circles)} circles | {len(imerg_times_np)} IMERG timesteps')

def _plot(circle_id):
    out  = OUT_DIR / f'circle_{int(circle_id):03d}.png'
    info = plot_one_circle(
        ds_sonde, df_cats, ds_imerg, imerg_times_np, cum_index,
        int(circle_id), out, category_col=CAT_COL,
    )
    print(f"  circle={info['circle']}  {info['category']}")
    print(f"  window : {info['circle_start']} – {info['circle_end']} UTC")
    print(f"  IMERG  : {info['n_imerg_panels']} panels")
    print(f"  saved  : {info['output']}")

In [ ]:
# ── Mode 1: Single circle ─────────────────────────────────────────────────────
CIRCLE = 1   # ← change circle number here

_plot(CIRCLE)

In [ ]:
# ── Mode 2: All circles on a specific date ────────────────────────────────────
DATE = '2024-08-11'   # ← change date here (YYYY-MM-DD)

target_date  = pd.to_datetime(DATE).date()
date_circles = [int(c) for c, ct in zip(circles, ctimes) if ct.date() == target_date]
print(f'{DATE}: {len(date_circles)} circles → {date_circles}')

for cid in date_circles:
    try:
        _plot(cid)
    except Exception as e:
        print(f'  circle={cid} ERROR: {e}')

In [ ]:
# ── Mode 3: All 89 circles (saves PNGs, no inline display) ───────────────────
saved, failed = 0, []

for cid in circles:
    out = OUT_DIR / f'circle_{int(cid):03d}.png'
    try:
        info = plot_one_circle(
            ds_sonde, df_cats, ds_imerg, imerg_times_np, cum_index,
            int(cid), out, category_col=CAT_COL,
        )
        print(f"  ✓ circle={int(cid):3d}  {info['category']:<35s}  {out.name}")
        saved += 1
    except Exception as e:
        print(f"  ✗ circle={int(cid):3d}  ERROR: {e}")
        failed.append(int(cid))

print(f'\nDone. Saved {saved}/{len(circles)} figures to {OUT_DIR}')
if failed:
    print(f'Failed: {failed}')